# C0 + C1 development evaluation — unattended Slurm jobs

**Suraj: open this one notebook in Sol Jupyter and run sections 1–3.**
Wait for the Slurm job IDs. You can then close your laptop and disconnect.
This evaluates the adapters that have already completed training.

| Item | Scope |
|---|---|
| C0 | 12 saved models, GSM + MATH development evaluation |
| C1 | 12 saved models, GSM + MATH development evaluation |
| Questions | 500 per benchmark, 1,000 per model |
| Reuse | Completed compatible pilot/parent/B predictions are checked and reused |
| New C0/C1 generation | Normally 23 models / 46 benchmark files |
| Training | None |
| Protected final tests | Reserved; not run |

The workflow produces all 12 C1-versus-C0 paired comparisons and parent
comparisons. The screened B pilot supports one matched three-method cell.
**Eleven further screened B controls remain before a full matched B/C0/C1
comparison.** Original B scores are imported as historical context, with their
different training bank disclosed.

A CPU Jupyter session of **one hour** is sufficient to submit. GPU jobs request
one A100 each, six serial four-hour allocations by default. **24 hours is the
total allocation cap, not an expected runtime.** Unused later allocations are
cancelled. Generation speed and queue time determine actual completion.


## 1. Configuration

Keep the existing training outputs, B runtime, models and environments in place.
No package installation or model download is performed.


In [ ]:
from pathlib import Path
WORK = Path('/scratch/sshind38/capstone_c0_c1_work')
B_ROOT = Path('/scratch/sshind38/capstone_approach_b')
PACKAGE_NAME = 'C0_C1_Development_Evaluation_20261110'
TRAIN_PYTHON = B_ROOT / 'venv_b_training_py312/bin/python'
ALLOCATIONS = 6
ACCOUNT = 'class_fse57081354fall2026'
PARTITION = 'public'
QOS = 'class'
print('Development evaluation ON. Training OFF. Protected final tests OFF.')
print('Existing C0/C1 training: 12 + 12 models. Compatible predictions will be reused.')


## 2. Install the embedded package

Run once. This verifies every embedded file and preserves any different
existing file. There is no separate ZIP to upload and no GPU work in this cell.


In [ ]:
import base64, hashlib, io, json, zipfile
_PAYLOAD_SHA256 = 'cf3afb591cfc1fb6aa06722caf8e2957411f7fb54e71b3cf0002e3ec9ac244a5'
_PAYLOAD = '...'
_blob = base64.b64decode(_PAYLOAD, validate=True)
if hashlib.sha256(_blob).hexdigest() != _PAYLOAD_SHA256:
    raise ValueError('Embedded package damaged; download the notebook again.')
if not WORK.is_dir() or not TRAIN_PYTHON.is_file():
    raise FileNotFoundError('Run on Sol with the existing project directories and B Python environment.')
_package = WORK / PACKAGE_NAME
with zipfile.ZipFile(io.BytesIO(_blob)) as _archive:
    if _archive.testzip():
        raise ValueError('Embedded ZIP checksum failed')
    _pins = json.loads(_archive.read('PACKAGE_PINS.json'))
    if set(_archive.namelist()) != set(_pins) | {'PACKAGE_PINS.json'}:
        raise ValueError('Unexpected embedded package members')
    for _member in _archive.infolist():
        _path = (_package / _member.filename).resolve()
        if not _path.is_relative_to(_package.resolve()):
            raise ValueError('Unsafe embedded path')
        _data = _archive.read(_member)
        if _member.filename in _pins and hashlib.sha256(_data).hexdigest() != _pins[_member.filename]:
            raise ValueError('Embedded file hash mismatch: ' + _member.filename)
        if _path.exists():
            if _path.read_bytes() != _data:
                raise FileExistsError('Different existing file preserved: ' + str(_path))
        else:
            _path.parent.mkdir(parents=True, exist_ok=True)
            with _path.open('xb') as _stream:
                _stream.write(_data)
print('Installed:', _package)
print('No jobs submitted yet. Run section 3.')


## 3. Launch — wait for job IDs

This verifies the completed training evidence, adapter weights/configurations,
offline model caches, two exact development datasets and frozen evaluator.
Compatible saved predictions are imported before Slurm submission.

Wait for **`submitted_evaluation_not_yet_verified` and the job IDs**.
After that, you can close Jupyter, Sol and the laptop. Pending jobs are queued.
Rerunning this cell reuses an active submission instead of creating duplicates.


In [ ]:
import subprocess
subprocess.run([str(TRAIN_PYTHON), '-u', '-B',
                str(WORK/PACKAGE_NAME/'evaluate_remaining.py'), 'submit',
                '--work', str(WORK), '--b-root', str(B_ROOT), '--allocations', str(ALLOCATIONS),
                '--account', ACCOUNT, '--partition', PARTITION, '--qos', QOS], cwd=WORK, check=True)


## 4. Live progress — run again whenever needed

This cell is independent and works after restarting the notebook kernel.
The **C0/C1 counters below mean evaluated models, not trained models**.
A model counts complete after both its 500-question evaluations pass.

There are 58 required files including parent/B helpers. Reused files already
count as complete. The current stage log shows live **16/500, 32/500, …**
question progress so you can see work while a model's completion count is unchanged.


In [ ]:
from pathlib import Path
import subprocess
_work = Path('/scratch/sshind38/capstone_c0_c1_work')
_broot = Path('/scratch/sshind38/capstone_approach_b')
subprocess.run([str(_broot/'venv_b_training_py312/bin/python'), '-u', '-B',
                str(_work/'C0_C1_Development_Evaluation_20261110/evaluate_remaining.py'), 'status',
                '--work', str(_work), '--b-root', str(_broot)], cwd=_work, check=True)


## 5. Collect the results after completion

Wait for **C0 12/12, C1 12/12, `evaluation_complete` and
`EVALUATION_COMPLETE.json`**. This cell verifies the final marker and creates
one handoff ZIP in your **Sol home folder**. It includes per-question outputs,
source/settings hashes, scores, paired comparisons and seed summaries, without weights.

Download the printed file using Sol Files → **Change directory** →
**`/home/sshind38/`** → select the ZIP → Download. Upload it to this conversation
for independent verification and interpretation of C0, C1 and available B results.


In [ ]:
from pathlib import Path
import json, subprocess
_work = Path('/scratch/sshind38/capstone_c0_c1_work')
_broot = Path('/scratch/sshind38/capstone_approach_b')
_package = _work/'C0_C1_Development_Evaluation_20261110'
_location = _package/'RUN_LOCATION.json'
if not _location.exists():
    print('Run the launch cell first.')
else:
    _root = Path(json.loads(_location.read_text())['control_directory'])
    if not (_root/'EVALUATION_COMPLETE.json').exists():
        print('Evaluation or comparison is still unfinished. Run section 4 for live progress.')
    else:
        subprocess.run([str(_broot/'venv_b_training_py312/bin/python'), '-u', '-B',
                        str(_package/'evaluate_remaining.py'), 'collect',
                        '--work', str(_work), '--b-root', str(_broot)], cwd=_work, check=True)


## 6. View the summary table after completion

`C0` and `C1` are the new screened study. `B_screened` is the matched pilot.
`B_legacy` is historical context from a different candidate bank. The paired
report computes C-minus-B effects only where a matched screened B exists.


In [ ]:
from pathlib import Path
import json, csv, html
from IPython.display import display, HTML
_package = Path('/scratch/sshind38/capstone_c0_c1_work/C0_C1_Development_Evaluation_20261110')
_location = _package/'RUN_LOCATION.json'
if not _location.exists():
    print('No submitted evaluation yet.')
else:
    _root = Path(json.loads(_location.read_text())['control_directory'])
    _table = _root/'reports/SEED_SUMMARY.csv'
    if not _table.exists():
        print('Summary is written after all required evaluations finish. Use section 4 meanwhile.')
    else:
        with _table.open() as _stream:
            _rows = list(csv.DictReader(_stream))
        _columns = ['model_size','parent_seed','method','seeds_completed','gsm_mean_percent',
                    'math_mean_percent','equal_weight_mean_percent','training_seed_SD_pp']
        _head = ''.join('<th>'+html.escape(c)+'</th>' for c in _columns)
        _body = ''.join('<tr>'+''.join('<td>'+html.escape(str(r.get(c,'')))+'</td>' for c in _columns)+'</tr>' for r in _rows)
        display(HTML('<table><thead><tr>'+_head+'</tr></thead><tbody>'+_body+'</tbody></table>'))
        print('Full report:', _root/'reports/comparison.md')
        print('Paired comparisons:', _root/'reports/PAIRED_COMPARISONS.csv')
        print('Eleven screened B controls remain before a full matched B/C0/C1 comparison.')


## Recovery and interpretation

- Native partial predictions resume under the same checkpoint, questions,
  settings and GPU platform. Completed results are hash checked and reused.
- Broken partial JSON attempts are preserved. No existing training output is
  changed, and no training or protected final-test job is submitted.
- If six allocations are exhausted, rerun section 3 for a new bounded chain.
  Completed results are reused. For blocked integrity/checker stages, inspect
  section 4 and the reported log; no settings are silently relaxed.
- Comparison files include accuracy, parsing/truncation/repetition, output
  length, paired fixed-checkpoint item-bootstrap intervals (2,000 draws), and
  descriptive three-seed mean/SD. They do not establish uncertainty of a
  three-seed population mean or automatically claim statistical significance.
- The report preserves inherited exposure and conservative-screen limitations.
  Development evidence does not certify contamination-free final generalization.

**Validation:** 24 local CPU tests, plus compatibility checks of all six real
pilot prediction files. GPU compute is simulated in orchestration tests.
No new Qwen evaluation or live Slurm submission has been executed here;
production source/checkpoint/runtime checks run when you launch on Sol.
